# 🏛️ CataList Meeting Intelligence Memory Agent
### **Agentic AI Intern: Build Test &bull; Interactive Google Colab Notebook**

Welcome to the official interactive notebook for the **CataList Meeting Intelligence Memory Engine**. This system turns **138 real-world AMI design meetings (~676,000 words, 48,148 dialogue turns across 35 teams)** into a searchable, temporally aware conversational memory that:
1. **Answers multi-meeting questions** in strictly 1–2 sentences with **100% verbatim citations**.
2. **Tracks decision evolution over time** across chronological phases ($a \rightarrow b \rightarrow c \rightarrow d$).
3. **Synthesizes pre-meeting executive briefs** (Team TS3012) with hard zero-leakage quarantine.
4. **Operates at $0.00 LLM ingestion cost** via SQLite FTS5 BM25 and 384-dimensional dense vectors.

---
## 📦 Step 1: Clone Repository & Install Dependencies
Run this cell first. It automatically clones the repository into your Colab runtime and installs the lightweight dependencies (`requirements.txt`).

In [ ]:
# @title 🚀 1. Clone Repo & Install Requirements
import os
from pathlib import Path

# Automatically clone if not already in the repository root
if not os.path.exists('src'):
    print('Cloning repository from GitHub...')
    !git clone https://github.com/kamranahmed0611/meeting-transcript-memory-AI-agent.git
    %cd meeting-transcript-memory-AI-agent

print('Installing dependencies from requirements.txt...')
!pip install -q -r requirements.txt
print('Environment setup complete!')

---
## 🔑 Step 2: Configure Your Google Gemini API Key
Enter your Google Gemini API key below.

> **How to get a free API Key:** Visit [Google AI Studio](https://aistudio.google.com/) and click **Get API Key**.
> 
> **Offline Evaluator Fallback:** If you leave the key blank, the agent automatically activates its local deterministic offline synthesizer (`Offline Granite + BM25 Fallback`), allowing you to test the entire notebook offline.

In [ ]:
# @title 🔑 2. Enter Gemini API Key
import os
import getpass

# Try loading from Colab Secrets if available
try:
    from google.colab import userdata
    secret_key = userdata.get('GEMINI_API_KEY')
except Exception:
    secret_key = None

# Enter your API key in the interactive input below
API_KEY_INPUT = "" # @param {type:"string"}

final_api_key = API_KEY_INPUT.strip() or secret_key or os.environ.get('GEMINI_API_KEY', '').strip()

if not final_api_key:
    # Prompt interactively if not provided in form
    print('👉 Paste your Gemini API key below (or press Enter to run in 100% Offline Mode):')
    try:
        prompted_key = getpass.getpass('Gemini API Key (optional): ').strip()
        if prompted_key:
            final_api_key = prompted_key
    except Exception:
        pass

if final_api_key:
    os.environ['GEMINI_API_KEY'] = final_api_key
    os.environ['GEMINI_MODEL'] = 'gemini-3.5-flash-lite'
    print('✅ Gemini API Key configured successfully!')
    print('   Active Model: gemini-3.5-flash-lite (Fastest, High Throughput)')
else:
    os.environ.pop('GEMINI_API_KEY', None)
    print('⚡ Offline Evaluator Mode Activated: Agent will run using deterministic Granite Embeddings + BM25.')

---
## ⚙️ Step 3: Initialize Memory Engine & Database Health Check
The memory engine will now check for `data/meeting_store.db`. If you just cloned the repository, the **self-healing pipeline** will automatically parse all 138 transcripts in `transcripts/` into 48,148 turns and build the SQLite FTS5 index and dense vector store in ~30 seconds.

In [ ]:
# @title ⚙️ 3. Initialize Meeting Agent & Database
import os
import sqlite3
from pathlib import Path

from src.agent import MeetingAgent
from src.verifier import CitationVerifier

# Initialize Agent (auto-triggers self-healing ingestion if db is missing)
print('Initializing CataList Meeting Agent...')
agent = MeetingAgent()
verifier = agent.verifier

# Verify SQLite Database Health
conn = sqlite3.connect('data/meeting_store.db')
cur = conn.cursor()
cur.execute('SELECT COUNT(*) FROM turns')
total_turns = cur.fetchone()[0]
cur.execute('SELECT COUNT(*) FROM meetings')
total_meetings = cur.fetchone()[0]
cur.execute('SELECT COUNT(*) FROM turn_embeddings')
total_embeddings = cur.fetchone()[0]
conn.close()

print(f'\n✅ Memory Database Status:')
print(f'   - Indexed Meetings:  {total_meetings} / 138 (100% AMI Corpus)')
print(f'   - Dialogue Turns:    {total_turns:,}')
print(f'   - Stored Embeddings: {total_embeddings:,}')
print(f'   - Synthesizer:       {"gemini-3.5-flash-lite (Online)" if agent.client else "Offline Granite + BM25 Fallback"}')

---
## 💬 Step 4: Ask Any Question (Interactive Meeting Query Console)
Type **any question** about meetings, components, price decisions, or feature trade-offs below.

**Suggested Questions to Try:**
* *Team TS3006 planned both an LCD screen and speech recognition. When the costs forced a cut, which of the two did they drop, which did they keep, and what reason was given for keeping it?*
* *What was Team ES2015's final operating cost total in meeting d, and what specific features did Speaker B note they had to sacrifice to stay within resources?*
* *What material did Team ES2003 choose for the remote casing?*
* *In Team TS3008, what component was fiercely debated regarding kinetic power in the final minutes?*

In [ ]:
# @title 🔍 Ask the Agent Any Meeting Question
# @markdown Type your question into the field below and run this cell:

user_question = "Team TS3006 planned both an LCD screen and speech recognition. When the costs forced a cut, which of the two did they drop, which did they keep, and what reason was given for keeping it?" # @param {type:"string"}

if not user_question.strip():
    user_question = input("Enter your question: ").strip()

print(f"\n🔎 Processing Query: \"{user_question}\"")
print("=" * 75)

# Step A: Deterministic Regex Team & Meeting Extraction
team_detected = agent.retriever.extract_team_id(user_question)
meeting_detected = agent.retriever.extract_meeting_id(user_question)
print(f"📍 Deterministic Scope Detection: Team={team_detected or 'Global'} | Meeting={meeting_detected or 'All'}")

# Step B: Run Agent Pipeline (Facet Decomposition -> Hybrid Retrieval -> Gap-Bridging -> Gemini Synthesis)
question_payload = {
    "question_id": "COLAB_USER_QUERY",
    "team": team_detected,
    "question": user_question
}

response = agent.answer_question(question_payload, verbose=False)

# Step C: Display Results with Telemetry & Exact Verbatim Lineproof Citations
print("\n🧠 SYNTHESIZED ANSWER (Strictly 1–2 Sentences):")
print("-" * 75)
print(response["answer"])
print("-" * 75)

print("\n📍 100% VERIFIED VERBATIM TRANSCRIPT CITATIONS:")
if response.get("citations"):
    for i, c in enumerate(response["citations"], 1):
        is_valid, exact_quote, turn = verifier.verify_quote(c["meeting_id"], c["quote"])
        status_badge = "✅ VERIFIED RAW SUBSTRING" if is_valid else "⚠️ UNVERIFIED"
        print(f"  [{i}] Meeting: [{c['meeting_id']}] | Status: {status_badge}")
        print(f"      Exact Words: \"{c['quote']}\"")
else:
    print("  (No citations returned for this general/unmentioned query)")

print(f"\n⚡ Telemetry: Retrieval={response.get('retrieval_ms', 0):.1f}ms | Bridged Context={response.get('bridged_turn_count', 0)} turns | Synthesizer={response.get('synthesizer')}")

---
## 📋 Step 5: Generate Team TS3012 Executive Briefing Memo
Generates the pre-meeting briefing that Team TS3012 should read before their final meeting ($d$).

> **Zero Future Leakage Guarantee:** This brief is synthesized strictly from meetings $a$, $b$, and $c$. Hard code assertions quarantine meeting $d$ ($0.0\%$ leakage).

In [ ]:
# @title 📋 5. Generate TS3012 Pre-Meeting Brief (0% Future Leakage)
from src.brief import BriefGenerator

brief_generator = BriefGenerator()
brief_markdown = brief_generator.generate_brief(client=agent.client)

print("=" * 75)
print(" TEAM TS3012 EXECUTIVE PRE-MEETING BRIEFING DOSSIER")
print("=" * 75)
print(brief_markdown)

---
## 🏆 Step 6: Run Automated Benchmark Auto-Grader
Run the master evaluation harness directly inside Colab. This grades all 25 official assessment questions and 10 custom proof-of-work questions against the ground truth, checking exact verbatim substring matches and sentence length limits.

In [ ]:
# @title 🏆 6. Run Benchmark Auto-Grader
!python eval_benchmarks.py